# Time Weighted Vector Store Retriever — 시간 가중치 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **TimeWeightedVectorStoreRetriever**의 시간 가중치 계산 방식 이해하기
- `decay_rate`로 시간 감소율을 조절하는 방법 익히기
- 뉴스, 대화 기록 등 시간에 민감한 데이터에 적용하기

## 사전 지식

- VectorStoreRetriever 기본 사용법
- 최신 정보와 오래된 정보의 관련성이 다를 수 있다는 도메인 지식

---

## 핵심 개념

### 일반 Vector Search vs Time Weighted Search

| 방식 | 점수 계산 |
|------|----------|
| 일반 검색 | 유사도만 고려 |
| **Time Weighted** | 유사도 × 시간 가중치 |

예시:
```
문서 A (1년 전, 유사도 0.9)  → 시간 감소 후: 0.6  (선택 안 됨)
문서 B (어제,   유사도 0.8)  → 시간 가중치: 0.9  (선택됨)
```

## 사용 시나리오

- 뉴스 피드 검색
- 실시간 정보 업데이트
- 대화 기록 기반 검색
- 시간에 민감한 금융·주식 데이터

> **실무 팁**: `decay_rate`를 아주 작은 값(예: 1e-24)으로 설정하면 사실상 시간 순서만 반영돼요. 반대로 0.9처럼 크게 설정하면 어제 데이터도 크게 감소합니다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# ---------------------------------------------------
# 시간 메타데이터가 다른 문서를 추가하고 최신 문서 우선 검색 확인
# ---------------------------------------------------

# ============================================================
# TODO: TimeWeightedVectorStoreRetriever를 생성하고 시간차를 둔 문서를 추가하세요
# 힌트: decay_rate가 작을수록 시간 감소 속도가 느림 (최신 문서 우대 효과 약함)
# 예상 결과: 검색 결과에서 최신 문서가 상위에 위치
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from datetime import datetime, timedelta
import time

# Vectorstore 준비
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_texts(
    ["빈 초기화용"],
    embedding=embeddings
)

# TimeWeightedVectorStoreRetriever 생성
# decay_rate: 시간 감소율 — 매우 작은 값이면 사실상 시간순 정렬


# 시간차를 두고 문서 추가 (last_accessed_at으로 최신성 지정)
docs_with_time = [
    ("오래된 정보: Scikit-learn은 2007년 시작되었습니다", 72),  # 72시간 전
    ("중간 정보: NLP는 자연어 처리 기술입니다", 8),              # 8시간 전
    ("최신 정보: ChatGPT는 2022년에 출시되었습니다", 0)          # 현재
]

for content, hours_ago in docs_with_time:
    time.sleep(0.1)
    doc = Document(
        page_content=content,
        metadata={"last_accessed_at": datetime.now() - timedelta(hours=hours_ago)}
    )
    time_weighted_retriever.add_documents([doc])

print("✅ TimeWeightedVectorStoreRetriever 생성 완료")

In [ ]:
# 검색
query = "기술 정보"
docs = time_weighted_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print("="*80)
print("[Time Weighted 결과]")
print("="*80)
for i, doc in enumerate(docs, 1):
    print(f"[{i}] {doc.page_content}")
print()

print("💡 특징:")
print("  - 최신 문서가 우선 순위 높음")
print("  - decay_rate로 시간 감소 속도 조절")
print("  - 시간에 민감한 정보 검색에 적합")


---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 최종 점수 = 의미 유사도 × (1 - decay_rate)^경과시간(시간 단위) |
| `decay_rate` 역할 | 값이 클수록 오래된 문서 점수가 빠르게 낮아짐 |
| 문서 접근 갱신 | 검색될 때마다 `last_accessed_at` 자동 갱신 |
| 적합한 용도 | 뉴스, 실시간 데이터, 날짜 민감한 FAQ |
| 주의 | 날짜 무관한 정보엔 비적합 — 유사도만 순수하게 쓰는 게 나을 수 있음 |

### `decay_rate` 선택 가이드

| `decay_rate` | 반감 속도 | 적합한 상황 |
|--------------|-----------|-------------|
| 0.01 | 약 70일 | 월간 업데이트 문서 |
| 0.1 | 약 7일 | 주간 뉴스, 블로그 |
| 0.5 | 약 1.4일 | 실시간 이슈 Q&A |
| 0.99 | 거의 즉시 | 당일 데이터만 유효한 경우 |

### 다음 노트북 예고

**10-Kiwi-BM25Retriever** — 한국어 형태소 분석기 Kiwi를 BM25와 결합해 한국어 키워드 검색 정확도를 높이는 방법을 배워요.